# Pediatric Critical Care Analytics — Phase 2: Diagnostic Analytics

**Phase 1 (descriptive)** told us *what* happened: mortality was 7.2%, LOS varies by unit, certain
diagnosis chapters dominate admissions. **Phase 2 (this notebook)** asks *why* — testing whether
the patterns we observed are statistically real, and which factors actually explain them, rather
than eyeballing a chart and guessing.

This is the layer that differentiates an analytics partner from a reporting tool: descriptive
dashboards tell a hospital *that* mortality differs by unit; diagnostic analytics tells them
**whether that difference is real, and whether it's explained by case-mix or by something
operational** — which is what leadership actually needs to act on.

## Dataset
Same source as Phase 1: **PIC (Paediatric Intensive Care) database**, PhysioNet v1.1.0.
Zeng, X. *et al.*, *Scientific Data* 7, 14 (2020). https://doi.org/10.1038/s41597-020-0355-4
Reference/demo data — not the target hospital's own data. Same four tables used in Phase 1:
`ADMISSIONS`, `ICUSTAYS`, `PATIENTS`, `DIAGNOSES_ICD`.

## Diagnostic questions this notebook answers

| # | Question | Method |
|---|---|---|
| 1 | Is the mortality difference between care units statistically real, or noise? | Chi-square test of independence |
| 2 | Is that mortality difference explained by case-mix (age, diagnosis), or does the unit itself still matter after adjusting? | Multivariate logistic regression (odds ratios) |
| 3 | What actually drives the spread in ICU length of stay — unit, age, or diagnosis? | Kruskal-Wallis test + effect size (eta-squared) |
| 4 | What distinguishes patients who get readmitted within 48h from those who don't? | Group comparison (chi-square / Mann-Whitney) + logistic regression |
| 5 | Are outcomes worse for admissions/discharges at certain times (weekend/night effect)? | Time-pattern breakdown + significance test |

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Load & rebuild the cohort (same as Phase 1)

In [24]:
admissions = pd.read_csv("ADMISSIONS.csv")
icustays = pd.read_csv("ICUSTAYS.csv")
patients = pd.read_csv("PATIENTS.csv")
diagnoses = pd.read_csv("DIAGNOSES_ICD.csv")

for col in ["ADMITTIME", "DISCHTIME", "DEATHTIME", "EDREGTIME", "EDOUTTIME"]:
    admissions[col] = pd.to_datetime(admissions[col], format="%d-%m-%Y %H:%M", errors="coerce")
for col in ["INTIME", "OUTTIME"]:
    icustays[col] = pd.to_datetime(icustays[col], errors="coerce")
for col in ["DOB", "DOD"]:
    patients[col] = pd.to_datetime(patients[col], errors="coerce")

cohort = (
    icustays
    .merge(admissions, on=["SUBJECT_ID", "HADM_ID"], how="left")
    .merge(patients, on="SUBJECT_ID", how="left")
)
cohort["age_years"] = (cohort["ADMITTIME"] - cohort["DOB"]).dt.days / 365.25
cohort["los_icu_days"] = cohort["LOS"]
cohort["los_hosp_days"] = (cohort["DISCHTIME"] - cohort["ADMITTIME"]).dt.total_seconds() / 86400

def age_group(age):
    if pd.isna(age): return "Unknown"
    if age < 1/12: return "Neonate (<1mo)"
    elif age < 1: return "Infant (1-12mo)"
    elif age < 3: return "Toddler (1-3yr)"
    elif age < 12: return "Child (3-12yr)"
    else: return "Adolescent (12-18yr)"

cohort["age_group"] = cohort["age_years"].apply(age_group)
age_order = ["Neonate (<1mo)", "Infant (1-12mo)", "Toddler (1-3yr)", "Child (3-12yr)", "Adolescent (12-18yr)"]

# Primary diagnosis chapter per admission (first-listed diagnosis, SEQ_NUM == 1 where available)
icd10_chapters = {
    "A": "Infectious & parasitic", "B": "Infectious & parasitic", "C": "Neoplasms",
    "D": "Blood / immune", "E": "Endocrine/metabolic", "F": "Mental/behavioural",
    "G": "Nervous system", "H": "Eye / ear", "I": "Circulatory system",
    "J": "Respiratory system", "K": "Digestive system", "L": "Skin",
    "M": "Musculoskeletal", "N": "Genitourinary system", "O": "Pregnancy/childbirth",
    "P": "Perinatal conditions", "Q": "Congenital malformations",
    "R": "Symptoms/unclassified", "S": "Injury/poisoning", "T": "Injury/poisoning",
    "Z": "Health status/services",
}
diagnoses["chapter_code"] = diagnoses["ICD10_CODE_CN"].astype(str).str[0].str.upper()
diagnoses["chapter_name"] = diagnoses["chapter_code"].map(icd10_chapters).fillna("Other/uncoded")

primary_diag = (
    diagnoses.sort_values(["HADM_ID", "SEQ_NUM"])
    .groupby("HADM_ID")
    .first()[["chapter_name"]]
    .rename(columns={"chapter_name": "primary_diagnosis_chapter"})
)
cohort = cohort.merge(primary_diag, on="HADM_ID", how="left")
cohort["primary_diagnosis_chapter"] = cohort["primary_diagnosis_chapter"].fillna("Not coded")

cohort["admit_dow"] = cohort["ADMITTIME"].dt.day_name()
cohort["admit_hour"] = cohort["ADMITTIME"].dt.hour
cohort["discharge_dow"] = cohort["DISCHTIME"].dt.day_name()
cohort["is_weekend_admit"] = cohort["admit_dow"].isin(["Saturday", "Sunday"])
cohort["is_night_admit"] = cohort["admit_hour"].between(0, 6) | (cohort["admit_hour"] >= 22)

print(f"Cohort: {len(cohort):,} ICU stays, {cohort['SUBJECT_ID'].nunique():,} patients")
cohort[["FIRST_CAREUNIT", "age_group", "primary_diagnosis_chapter",
        "los_icu_days", "HOSPITAL_EXPIRE_FLAG"]].head()


Cohort: 13,941 ICU stays, 12,881 patients


,FIRST_CAREUNIT,age_group,primary_diagnosis_chapter,los_icu_days,HOSPITAL_EXPIRE_FLAG
0,General ICU,Infant (1-12mo),Digestive system,4.6167,0
1,General ICU,Child (3-12yr),Circulatory system,9.1771,1
2,General ICU,Toddler (1-3yr),Injury/poisoning,7.7326,0
3,General ICU,Child (3-12yr),Not coded,4.4132,1
4,General ICU,Infant (1-12mo),Digestive system,1.7479,1


## 2. Diagnostic Q1 — Is the mortality gap between care units statistically real?

Phase 1 showed crude mortality varying by unit. Before treating that as a quality signal, we test
whether it's more than random noise, **and** whether it survives adjustment for who's actually
being treated in each unit (case-mix).


In [25]:
ct = pd.crosstab(cohort["FIRST_CAREUNIT"], cohort["HOSPITAL_EXPIRE_FLAG"])
chi2, p_val, dof, expected = stats.chi2_contingency(ct)

print(f"Chi-square test: care unit vs. mortality")
print(f"  chi2 = {chi2:.2f}, dof = {dof}, p-value = {p_val:.2e}")
if p_val < 0.05:
    print("  -> Statistically significant: mortality really does differ by care unit (p < 0.05).")
else:
    print("  -> Not statistically significant: observed differences could be due to chance.")

display(ct.assign(mortality_rate_pct=(ct[1] / (ct[0]+ct[1]) * 100).round(2)))


Chi-square test: care unit vs. mortality
  chi2 = 510.82, dof = 4, p-value = 3.06e-109
  -> Statistically significant: mortality really does differ by care unit (p < 0.05).


HOSPITAL_EXPIRE_FLAG,0,1,mortality_rate_pct
FIRST_CAREUNIT,,,
CICU,2744,59,2.1000
General ICU,2364,424,15.2100
NICU,3035,247,7.5300
PICU,1958,208,9.6000
SICU,2841,61,2.1000


In [26]:
# Is the unit effect explained by case-mix (age, diagnosis), or does unit still matter after adjusting?
model_df = cohort.dropna(subset=["age_years", "FIRST_CAREUNIT", "primary_diagnosis_chapter", "HOSPITAL_EXPIRE_FLAG"]).copy()

# Keep diagnosis chapters with reasonable sample size to avoid unstable dummy coefficients
top_chapters = model_df["primary_diagnosis_chapter"].value_counts()
keep_chapters = top_chapters[top_chapters >= 50].index
model_df["diag_group"] = np.where(
    model_df["primary_diagnosis_chapter"].isin(keep_chapters),
    model_df["primary_diagnosis_chapter"], "Other (low volume)"
)

logit_model = smf.logit(
    "HOSPITAL_EXPIRE_FLAG ~ C(FIRST_CAREUNIT) + age_years + C(diag_group)",
    data=model_df
).fit(disp=False)

print(logit_model.summary())

odds_ratios = np.exp(logit_model.params).rename("odds_ratio")
conf = np.exp(logit_model.conf_int())
conf.columns = ["ci_low", "ci_high"]
or_table = pd.concat([odds_ratios, conf, logit_model.pvalues.rename("p_value")], axis=1)

print("\nCare-unit odds ratios for mortality (relative to reference unit), adjusted for age + diagnosis:")
display(or_table.filter(like="FIRST_CAREUNIT", axis=0).sort_values("odds_ratio", ascending=False))


                            Logit Regression Results                            
Dep. Variable:     HOSPITAL_EXPIRE_FLAG   No. Observations:                13941
Model:                            Logit   Df Residuals:                    13917
Method:                             MLE   Df Model:                           23
Date:                  Thu, 01 Oct 2026   Pseudo R-squ.:                  0.1024
Time:                          18:12:34   Log-Likelihood:                -3227.3
converged:                         True   LL-Null:                       -3595.5
Covariance Type:              nonrobust   LLR p-value:                9.635e-141
                                                coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------
Intercept                                    -3.5753      0.218    -16.429      0.000      -4.002      -3.149
C(FIRST_CAREUNIT)[T.Ge

,odds_ratio,ci_low,ci_high,p_value
C(FIRST_CAREUNIT)[T.General ICU],9.2743,6.7601,12.7234,0.0000
C(FIRST_CAREUNIT)[T.PICU],4.7517,3.4123,6.6170,0.0000
C(FIRST_CAREUNIT)[T.NICU],3.2121,2.2835,4.5184,0.0000
C(FIRST_CAREUNIT)[T.SICU],1.0982,0.7500,1.6080,0.6303


**How to read this:** if a care unit's adjusted odds ratio is still significantly >1 (p<0.05)
*after* controlling for age and diagnosis mix, that's a genuine signal worth investigating
operationally — not just "sicker patients go there." If the raw crude difference from Section 2's
crosstab shrinks toward 1 once adjusted, the unit-level mortality gap is mostly explained by
case-mix, not care quality.

## 3. Diagnostic Q2 — What actually drives the spread in ICU length of stay?

Kruskal-Wallis (non-parametric ANOVA) tests whether LOS distributions genuinely differ across
groups, and **eta-squared** tells us how much of the total LOS variance each factor explains —
critical for prioritizing which lever (unit protocols vs. case-mix vs. age-based care pathways)
is worth intervening on first.


In [27]:
def kruskal_eta_squared(df, group_col, value_col):
    groups = [g[value_col].dropna().values for _, g in df.groupby(group_col)]
    groups = [g for g in groups if len(g) > 0]
    h_stat, p_val = stats.kruskal(*groups)
    n = sum(len(g) for g in groups)
    k = len(groups)
    eta_sq = (h_stat - k + 1) / (n - k) if n > k else np.nan
    return h_stat, p_val, eta_sq

results = []
for factor in ["FIRST_CAREUNIT", "age_group", "primary_diagnosis_chapter"]:
    h, p, eta = kruskal_eta_squared(cohort, factor, "los_icu_days")
    results.append({"factor": factor, "H_statistic": round(h, 1), "p_value": p,
                     "eta_squared": round(eta, 4)})

los_drivers = pd.DataFrame(results).sort_values("eta_squared", ascending=False)
print("Which factor explains the most variance in ICU LOS?")
print("(eta-squared: rough guide — 0.01 small, 0.06 medium, 0.14+ large effect)\n")
display(los_drivers)


Which factor explains the most variance in ICU LOS?
(eta-squared: rough guide — 0.01 small, 0.06 medium, 0.14+ large effect)



,factor,H_statistic,p_value,eta_squared
2,primary_diagnosis_chapter,"3,378.2000",0.0000,0.2413
0,FIRST_CAREUNIT,"3,321.9000",0.0000,0.2381
1,age_group,"1,666.4000",0.0000,0.1193


## 4. Diagnostic Q3 — What distinguishes 48h-readmission patients from the rest?

Root-causing the readmission flag from Phase 1: are readmitted patients systematically younger,
have a different diagnosis mix, shorter index-stay LOS (possible premature discharge), or come
from a particular unit?


In [28]:
stays_sorted = cohort.sort_values(["SUBJECT_ID", "INTIME"]).copy()
stays_sorted["prev_outtime"] = stays_sorted.groupby("SUBJECT_ID")["OUTTIME"].shift(1)
stays_sorted["gap_hours"] = (stays_sorted["INTIME"] - stays_sorted["prev_outtime"]).dt.total_seconds() / 3600
stays_sorted["readmit_48h"] = stays_sorted["gap_hours"].between(0, 48)

readmit_ids = set(stays_sorted.loc[stays_sorted["readmit_48h"], "SUBJECT_ID"])
cohort["was_readmitted_48h"] = cohort["SUBJECT_ID"].isin(readmit_ids)

print(f"Patients with an unplanned 48h ICU readmission: {len(readmit_ids)} "
      f"({len(readmit_ids)/cohort['SUBJECT_ID'].nunique()*100:.2f}% of patients)\n")

# Compare index-stay LOS: readmitted vs not
readmit_los = cohort.loc[cohort["was_readmitted_48h"], "los_icu_days"].dropna()
non_readmit_los = cohort.loc[~cohort["was_readmitted_48h"], "los_icu_days"].dropna()
u_stat, p_los = stats.mannwhitneyu(readmit_los, non_readmit_los, alternative="two-sided")

print(f"Index-stay LOS — readmitted median: {readmit_los.median():.2f}d, "
      f"non-readmitted median: {non_readmit_los.median():.2f}d")
print(f"Mann-Whitney U test p-value: {p_los:.4f} "
      f"{'(significant difference)' if p_los < 0.05 else '(no significant difference)'}")

# Compare care unit and diagnosis chapter distribution
print("\nCare unit — readmitted vs. overall population:")
compare_unit = pd.concat([
    cohort.loc[cohort["was_readmitted_48h"], "FIRST_CAREUNIT"].value_counts(normalize=True).rename("readmitted_%") * 100,
    cohort["FIRST_CAREUNIT"].value_counts(normalize=True).rename("overall_%") * 100,
], axis=1).round(1)
display(compare_unit)


Patients with an unplanned 48h ICU readmission: 197 (1.53% of patients)

Index-stay LOS — readmitted median: 6.14d, non-readmitted median: 2.73d
Mann-Whitney U test p-value: 0.0000 (significant difference)

Care unit — readmitted vs. overall population:


,readmitted_%,overall_%
FIRST_CAREUNIT,,
CICU,27.0000,20.1000
NICU,22.6000,23.5000
PICU,20.6000,15.5000
SICU,16.1000,20.8000
General ICU,13.6000,20.0000


## 5. Diagnostic Q4 — Is there a weekend or night-admission effect?

A common hypothesis in hospital quality analytics: outcomes may be worse for admissions/discharges
at night or on weekends due to lower staffing coverage — the classic "weekend effect." Worth
testing before assuming it, since it's often smaller than intuition suggests.


In [29]:
weekend_mortality = cohort.groupby("is_weekend_admit")["HOSPITAL_EXPIRE_FLAG"].agg(
    admissions="count", deaths="sum"
)
weekend_mortality["mortality_rate_%"] = (weekend_mortality["deaths"] / weekend_mortality["admissions"] * 100).round(2)
weekend_mortality.index = weekend_mortality.index.map({True: "Weekend admission", False: "Weekday admission"})

night_mortality = cohort.groupby("is_night_admit")["HOSPITAL_EXPIRE_FLAG"].agg(
    admissions="count", deaths="sum"
)
night_mortality["mortality_rate_%"] = (night_mortality["deaths"] / night_mortality["admissions"] * 100).round(2)
night_mortality.index = night_mortality.index.map({True: "Night admission (10pm-6am)", False: "Day admission"})

print("Weekend vs. weekday admission mortality:")
display(weekend_mortality)
ct_weekend = pd.crosstab(cohort["is_weekend_admit"], cohort["HOSPITAL_EXPIRE_FLAG"])
chi2_w, p_w, _, _ = stats.chi2_contingency(ct_weekend)
print(f"Chi-square p-value: {p_w:.4f} {'(significant)' if p_w < 0.05 else '(not significant)'}\n")

print("Night vs. day admission mortality:")
display(night_mortality)
ct_night = pd.crosstab(cohort["is_night_admit"], cohort["HOSPITAL_EXPIRE_FLAG"])
chi2_n, p_n, _, _ = stats.chi2_contingency(ct_night)
print(f"Chi-square p-value: {p_n:.4f} {'(significant)' if p_n < 0.05 else '(not significant)'}")


Weekend vs. weekday admission mortality:


,admissions,deaths,mortality_rate_%
is_weekend_admit,,,
Weekday admission,11731,780,6.6500
Weekend admission,2210,219,9.9100


Chi-square p-value: 0.0000 (significant)

Night vs. day admission mortality:


,admissions,deaths,mortality_rate_%
is_night_admit,,,
Day admission,12515,843,6.7400
Night admission (10pm-6am),1426,156,10.9400


Chi-square p-value: 0.0000 (significant)
